In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, lit, current_timestamp, coalesce, when, row_number, desc, concat_ws, trim
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("business_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primary.exposure")
        .filter(col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

In [0]:
# ==================== Load Source Tables ====================

# Safe date literal for column comparison
run_date_lit = lit(run_date).cast("date")

# Party record: run-date snapshot, deduped by CDI on source priority
# (profits > qca > other) then latest BusinessDate. This picks the CANONICAL
# record row - names, BirthDate, PartyType, PartyKey(cust_id). The tax number is
# resolved SEPARATELY below (review #4), so the tax tie-break does not drag the
# whole record to a lower-priority source.
party_all = (
    spark.table(f"`{env}-cdh`.primary.party")
    .filter(col("BusinessDate") == run_date_lit)
)

party_ranked = party_all.withColumn(
    "_src_priority",
    when(col("SourceSystem") == "profits", 1)
    .when(col("SourceSystem") == "qca", 2)
    .otherwise(3)
)

w = Window.partitionBy("CDI").orderBy(
    col("_src_priority"),
    col("BusinessDate").desc()
)
party = (
    party_ranked
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn", "_src_priority")
)

# Tax lookup (review #4/#2): CDI -> non-blank TaxRegistrationNumber, resolved
# INDEPENDENTLY of the record dedup. Run-date scoped (same snapshot as the record;
# all-history adds only ~13 CDIs). Ordering: non-blank tax first, then source
# priority, then latest date. Blank/whitespace tax normalised to NULL (review #3).
# Used ONLY for CustomerVat and ParentPartyId. Kept consistent with
# portfolio_cdi_bal.party_tax.
party_tax = (
    party_all.select(
        "CDI", "SourceSystem", "BusinessDate",
        when(trim(col("TaxRegistrationNumber")) != "", trim(col("TaxRegistrationNumber")))
            .alias("tax")
    )
    .withColumn(
        "_tax_rn",
        row_number().over(
            Window.partitionBy("CDI").orderBy(
                when(col("tax").isNotNull(), 0).otherwise(1),
                when(col("SourceSystem") == "profits", 1)
                .when(col("SourceSystem") == "qca", 2)
                .otherwise(3),
                col("BusinessDate").desc(),
                col("tax")
            )
        )
    )
    .filter(col("_tax_rn") == 1)
    .select("CDI", "tax")
)

# Exposure perimeter: distinct CDIs with active loan relationships for the run_date
# Using get_all_exposures TVF
exposure_perimeter = (
    spark.sql(f"""
        SELECT DISTINCT CDI
        FROM `{env}-cdh`.primary.get_all_exposures('{run_date}')
        WHERE PartyRelationshipType IN ('Guarantor', 'Primary Owner', 'Co-Owner')
    """)
)

# IdentityCard
id_card = (
    spark.table(f"`{env}-cdh`.primary.partyidentification")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & col("PartyIdentificationType").isin(
            "IdentityCard", "ΔΕΛΤΙΟ ΑΣΤΥΝΟΜΙΚΗΣ ΤΑΥΤΟΤΗΤΑΣ"
        )
    )
    .select(
        "CDI",
        col("PartyIdentificationType").alias("ID_CARD_TYPE"),
        col("PartyIdentificationNumber").alias("ID_CARD_NUMERIC"),
    )
    .distinct()
)

# PartyIdQualco: PartyKey from QCA source for same CDI
party_qca = (
    spark.table(f"`{env}-cdh`.primary.party")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("SourceSystem") == "qca")
    )
    .select("CDI", col("PartyKey").alias("PartyIdQualco"))
    .distinct()
)

# UnderLaw 3869: parties with active InLawFlag
underlaw_3869 = (
    spark.table(f"`{env}-cdh`.primary.underlaw")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("UnderLawType") == "N3869")
        & (col("InLawFlag") == 1)
    )
    .select("CDI")
    .distinct()
    .withColumn("IsL3869Flag", lit(1))
)

representative_keys = (
    party_all
    .filter(
        col("RepresentedByPartyKey").isNotNull()
        & (col("RepresentedByPartyType") == "REPRESENTS")  
    )
    .select(col("RepresentedByPartyKey").alias("RepKey"))
    .distinct()
)

In [0]:
# ==================== dtransfer_customer ====================
# Perimeter: parties present in exposure with Guarantor/Primary Owner/Co-Owner relationships

customer_df = (
    party
    .join(exposure_perimeter, on="CDI", how="inner")
    .join(id_card, on="CDI", how="left")
    .join(party_qca, on="CDI", how="left")
    .join(underlaw_3869, on="CDI", how="left")
    .join(party_tax, on="CDI", how="left")
    .select(
        current_timestamp().alias("ETLLoadingTime"),
        lit(run_date).cast("date").alias("ReferenceDate"),
        # PartyId = CDI (agreed cross-platform rule: DWH PartyId has no cloud
        # equivalent; CDI is the unique, non-colliding party identifier). When both
        # CDI and PartyId columns exist we populate them with the same value.
        # IT confirmed the alphanumeric string domain for PartyId/ParentPartyId is accepted.
        party["CDI"].alias("PartyId"),
        col("tax").alias("CustomerVat"),
        party["CDI"],
        coalesce(
            party["LegalNameGR"],
            when(party["PartyType"] == "Individual",
                 concat_ws(" ", party["FirstNameGR"], party["LastNameGR"]))
        ).alias("PartyName"),
        party["PartyType"].alias("cust_type"),
        party["LastNameGR"].alias("LAST_NAME"),
        when(party["PartyType"] == "Organization", party["PrimaryCommercialGR"]).alias("COMPANY_NAME"),
        party["FirstNameGR"].alias("FIRST_NAME"),
        party["FirstFatherNameGR"].alias("FATHER_NAME"),
        when(col("ID_CARD_NUMERIC").isNotNull(), col("ID_CARD_TYPE")).alias("ID_CARD_TYPE"),
        coalesce(col("ID_CARD_NUMERIC"), lit("ΑΓΝΩΣΤΟ")).alias("ID_CARD_NUMERIC"),
        party["PartyKey"].alias("cust_id"),
        # ParentPartyId = COALESCE(party_tax.tax, CDI) (agreed rule; CDI
        # fallback guarantees non-null since no DWH ParentPartyId exists in cloud).
        coalesce(col("tax"), party["CDI"]).alias("ParentPartyId"),
        coalesce(col("IsL3869Flag"), lit(0)).cast("boolean").alias("IsL3869Flag"),
        lit(None).cast("string").alias("Profits3869"),
        party["BirthDate"].cast("date").alias("BirthDate"),
        lit(None).cast("string").alias("PartyIdQualco"),
    )
    .distinct()
)

print(f"dtransfer_customer rows: {customer_df.count():,}")

In [0]:
# ==================== Write dtransfer_customer ====================

target_customer = f"`{env}-cdh-delivery`.it.dtransfer_customer"

(
    customer_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_customer)
)

print(f"Written to {target_customer}")

In [0]:
# ==================== dtransfer_representatives ====================
# Perimeter: parties that represent other parties (RepresentedByPartyKey IS NOT NULL)

representatives_df = (
    party
    #.filter(col("RepresentedByPartyKey").isNotNull())
    .join(representative_keys, party["PartyKey"] == representative_keys["RepKey"], how="inner")
    .drop("RepKey")
    .join(id_card, on="CDI", how="left")
    .join(party_qca, on="CDI", how="left")
    .join(underlaw_3869, on="CDI", how="left")
    .join(party_tax, on="CDI", how="left")
    .select(
        current_timestamp().alias("ETLLoadingTime"),
        lit(run_date).cast("date").alias("ReferenceDate"),
        # PartyId = CDI (agreed cross-platform rule). Populated identically to the
        # CDI column below so downstream Customer.PartyId joins stay consistent.
        # IT confirmed the alphanumeric string domain for PartyId/ParentPartyId is accepted.
        party["CDI"].alias("PartyId"),
        col("tax").alias("CustomerVat"),
        party["CDI"],
        coalesce(
            party["LegalNameGR"],
            when(party["PartyType"] == "Individual",
                 concat_ws(" ", party["FirstNameGR"], party["LastNameGR"]))
        ).alias("PartyName"),
        party["LastNameGR"].alias("LAST_NAME"),
        when(party["PartyType"] == "Organization", party["PrimaryCommercialGR"]).alias("COMPANY_NAME"),
        party["FirstNameGR"].alias("FIRST_NAME"),
        party["FirstFatherNameGR"].alias("FATHER_NAME"),
        when(col("ID_CARD_NUMERIC").isNotNull(), col("ID_CARD_TYPE")).alias("ID_CARD_TYPE"),
        coalesce(col("ID_CARD_NUMERIC"), lit("ΑΓΝΩΣΤΟ")).alias("ID_CARD_NUMERIC"),
        lit("REPRESENTS").alias("cust_type"),  
        # ParentPartyId = COALESCE(party_tax.tax, CDI) (agreed rule).
        coalesce(col("tax"), party["CDI"]).alias("ParentPartyId"),
        coalesce(col("IsL3869Flag"), lit(0)).cast("boolean").alias("IsL3869Flag"),
        lit(None).cast("string").alias("Profits3869"),
        party["BirthDate"].cast("date").alias("BirthDate"),
        lit(None).cast("string").alias("PartyIdQualco"),
    )
    .distinct()
)

print(f"dtransfer_representatives rows: {representatives_df.count():,}")

In [0]:
# ==================== Write dtransfer_representatives ====================

target_representatives = f"`{env}-cdh-delivery`.it.dtransfer_representatives"

(
    representatives_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_representatives)
)

print(f"Written to {target_representatives}")